In [1]:
import json
import zipfile
from collections import Counter

# 1. Load JSONL from ZIP
def load_zip_jsonl(path):
    with zipfile.ZipFile(path, 'r') as zf:
        name = zf.namelist()[0]

        data = []
        with zf.open(name) as f:
            for line in f:
                line = line.decode("utf-8").strip()
                if line:
                    data.append(json.loads(line))
    return data


# 2. Load dataset
path = "/Users/alvinwang/Desktop/google_news_rss_withtime_mpi.jsonl.zip"
data = load_zip_jsonl(path)

print("Total records:", len(data))

# 3. Check schema consistency
all_keys = [set(d.keys()) for d in data]

unique_structures = len(set(frozenset(k) for k in all_keys))
print("Unique structures:", unique_structures)


# 4. Find mismatched records
base = set(data[0].keys())

for i, d in enumerate(data):
    if set(d.keys()) != base:
        print("\nMismatch found at index:", i)
        print("Missing keys:", base - set(d.keys()))
        print("Extra keys:", set(d.keys()) - base)
        break


# 5. Key frequency analysis
key_counts = Counter()

for d in data:
    for k in d.keys():
        key_counts[k] += 1

print("\nKey coverage:")
for k, v in key_counts.most_common():
    print(f"{k}: {v}/{len(data)}")


# 6. Sample record 
print("\nSample record:")
print(json.dumps(data[0], indent=2, ensure_ascii=False))

Total records: 12439
Unique structures: 1

Key coverage:
article_id: 12439/12439
source_api: 12439/12439
source_name: 12439/12439
title: 12439/12439
description: 12439/12439
published_at: 12439/12439
date: 12439/12439
query: 12439/12439
keyword: 12439/12439
rss_url: 12439/12439
google_news_url: 12439/12439
original_url: 12439/12439
final_url: 12439/12439
canonical_url: 12439/12439
full_text: 12439/12439
text_length: 12439/12439
contains_trump: 12439/12439
filter_status: 12439/12439
decode_method: 12439/12439
decode_error: 12439/12439
fetch_status: 12439/12439
http_status: 12439/12439
extract_method: 12439/12439
error: 12439/12439
crawl_time: 12439/12439
article_publish_fetch_http_status: 12439/12439
article_publish_fetch_url: 12439/12439
article_published_at: 12439/12439
article_published_raw: 12439/12439
article_publish_time_source: 12439/12439
article_publish_time_confidence: 12439/12439

Sample record:
{
  "article_id": "968586a006c378b4de8cc144fb83926d",
  "source_api": "GoogleNews

In [2]:
!pip install pandas
import pandas as pd


def load_zip_jsonl(path):
    with zipfile.ZipFile(path, 'r') as zf:
        name = zf.namelist()[0]

        data = []
        with zf.open(name) as f:
            for line in f:
                line = line.decode("utf-8").strip()
                if line:
                    data.append(json.loads(line))
    return data


# load
path = "/Users/alvinwang/Desktop/google_news_rss_withtime_mpi.jsonl.zip"
data = load_zip_jsonl(path)

df = pd.DataFrame(data)

# convert date
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# sort by date (ascending = earliest → latest)
df = df.sort_values("date")

df.head()

df.to_csv(
    "/Users/alvinwang/Desktop/news_source.csv",
    index=False,
    encoding="utf-8-sig"
)

In [7]:
!pip install numpy
import numpy as np
import pandas as pd
import sys
!{sys.executable} -m pip install nltk
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

lm_df = pd.read_csv("/Users/alvinwang/Desktop/Loughran-McDonald_MasterDictionary_1993-2025.csv")
lm_df["Word"] = lm_df["Word"].astype(str).str.lower()

lm_negative = lm_df[lm_df["Negative"] > 0]["Word"].dropna().tolist()
lm_positive = lm_df[lm_df["Positive"] > 0]["Word"].dropna().tolist()

hawkish_phrases = ["carrier threat", "threat escalation", "talks collapse", "airstrike"]
dovish_phrases = ["bombing pause", "dialogue improving", "peace talks"]

hawkish_single_words = [
    "attack", "strike", "war", "military", "sanction",
    "destroy", "threat", "escalate", "retaliation",
    "bomb", "invasion", "deploy"
] + lm_negative

dovish_single_words = [
    "peace", "ceasefire", "negotiation", "talks",
    "dialogue", "agreement", "deal", "diplomatic",
    "pause", "suspension", "calm", "stability"
] + lm_positive

negation_words = ["no", "not", "never", "stop", "pause", "cancel", "without"]
negation_stemmed = [stemmer.stem(w) for w in negation_words]

hawkish_singles_stemmed = list(set([stemmer.stem(w) for w in hawkish_single_words]))
dovish_singles_stemmed = list(set([stemmer.stem(w) for w in dovish_single_words]))

hawkish_phrases_under = [p.replace(" ", "_") for p in hawkish_phrases]
dovish_phrases_under = [p.replace(" ", "_") for p in dovish_phrases]

final_hawkish_dict = [stemmer.stem(p) for p in hawkish_phrases_under] + hawkish_singles_stemmed
final_dovish_dict = [stemmer.stem(p) for p in dovish_phrases_under] + dovish_singles_stemmed

def score_text_rigorous(text):
    if pd.isna(text):
        return 0

    text = text.lower()
    
    for p in hawkish_phrases:
        text = text.replace(p, p.replace(" ", "_"))
    for p in dovish_phrases:
        text = text.replace(p, p.replace(" ", "_"))
        
    raw_words = text.split()
    final_tokens = [stemmer.stem(w) for w in raw_words]
    
    hawk_count = 0
    dove_count = 0
    
    for idx, token in enumerate(final_tokens):
        is_hawk = token in final_hawkish_dict
        is_dove = token in final_dovish_dict
        
        if not is_hawk and not is_dove:
            continue
            
        start_idx = max(0, idx - 3)
        context_window = final_tokens[start_idx:idx]
        
        has_negation = any(neg in context_window for neg in negation_stemmed)
        
        if is_hawk:
            if has_negation:
                dove_count += 1
            else:
                hawk_count += 1
        elif is_dove:
            if has_negation:
                hawk_count += 1
            else:
                dove_count += 1

    return hawk_count - dove_count

df = pd.read_csv("/Users/alvinwang/Desktop/news_source.csv")

df = df.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date")

df["rhetoric_score"] = df["full_text"].apply(score_text_rigorous)

daily = df.groupby("date").agg(
    rhetoric_index=("rhetoric_score", "mean"),
    article_count=("article_id", "count")
).reset_index()

daily["rhetoric_index"] = daily["rhetoric_index"].round(3)

events = pd.DataFrame([
    ("2026-02-10", "Escalation of Aircraft Carrier Threat"),
    ("2026-02-26", "Collapse of the Geneva Diplomatic Negotiations"),
    ("2026-02-28", "Commencement of Joint U.S.-Israeli Military Strikes"),
    ("2026-03-02", "Opening of the Hezbollah Northern Front"),
    ("2026-03-09", "Trump's Statement on the Potential End of Conflict"),
    ("2026-03-23", "Trump's Statement on Improving Diplomatic Dialogue"),
    ("2026-04-06", "Trump's Issuance of the Ultimatum"),
    ("2026-04-07", "Trump's Announcement of Cessation of Airstrikes"),
    ("2026-04-30", "Brent Crude Oil Price Peak and Market Panic"),
    ("2026-05-05", "Stalemate Phase"),
], columns=["date", "event"])

events["date"] = pd.to_datetime(events["date"])

merged = daily.merge(events, on="date", how="left")

window_results = []

for _, ev in events.iterrows():
    event_date = ev["date"]

    window = daily[
        (daily["date"] >= event_date - pd.Timedelta(days=3)) &
        (daily["date"] <= event_date + pd.Timedelta(days=3))
    ].copy()

    window["event"] = ev["event"]
    window["event_date"] = event_date

    window_results.append(window)

event_window_df = pd.concat(window_results, ignore_index=True)

event_summary = event_window_df.groupby(["event_date", "event"]).agg(
    avg_rhetoric=("rhetoric_index", "mean"),
    volatility=("rhetoric_index", "std"),
    obs=("rhetoric_index", "count")
).reset_index()

event_summary = event_summary.rename(columns={"event_date": "date"})
event_summary = event_summary.sort_values("date").reset_index(drop=True)

event_summary["avg_rhetoric"] = event_summary["avg_rhetoric"].round(3)
event_summary["volatility"] = event_summary["volatility"].round(3)
event_window_df["rhetoric_index"] = event_window_df["rhetoric_index"].round(3)

daily.columns = daily.columns.str.lower()
event_window_df.columns = event_window_df.columns.str.lower()
event_summary.columns = event_summary.columns.str.lower()

print(daily.head())
print(event_summary)

daily.to_csv("/Users/alvinwang/Desktop/daily_rhetoric.csv", index=False)
event_window_df.to_csv("/Users/alvinwang/Desktop/event_window_rhetoric.csv", index=False)
event_summary.to_csv("/Users/alvinwang/Desktop/event_summary_rhetoric.csv", index=False)


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
        date  rhetoric_index  article_count
0 2026-02-01          14.140            107
1 2026-02-02          16.229             48
2 2026-02-03          20.000             47
3 2026-02-04          12.849             53
4 2026-02-05          14.159             44
        date                                              event  avg_rhetoric  \
0 2026-02-10              Escalation of Aircraft Carrier Threat        13.988   
1 2026-02-26     Collapse of the Geneva Diplomatic Negotiations        28.071   
2 2026-02-28  Commencement of Joint U.S.-Israeli Military St...        29.152   
3 2026-03-02            Opening of the Hezbollah Northern Front        30.190   
4 2026-03-09  Trump's Statement on the Potential End of Conf...        26.099   
5 2026-03-23  Trump's Statement on Improving Diplomatic Dial...        24.003   
6 2026-04-06                  Trump's Issuance of the Ult

In [ ]:
# Alter the Format of Daily Oil Price
import pandas as pd

daily_oil_price = pd.read_csv("/Users/alvinwang/Desktop/daily_oil_price.csv")

daily_oil_price.columns = daily_oil_price.columns.str.lower()

target_columns = ["open", "low", "high", "close"]
daily_oil_price[target_columns] = daily_oil_price[target_columns].round(3)

daily_oil_price.to_csv("/Users/alvinwang/Desktop/daily_oil_price.csv", index=False)

In [12]:
import pandas as pd

daily_oil_price = pd.read_csv("/Users/alvinwang/Desktop/daily_oil_price.csv")
event_window = pd.read_csv("/Users/alvinwang/Desktop/event_window_rhetoric.csv")

daily_oil_price["date"] = pd.to_datetime(daily_oil_price["date"], errors="coerce", utc=True).dt.strftime("%Y-%m-%d")
event_window["date"] = pd.to_datetime(event_window["date"], errors="coerce", utc=True).dt.strftime("%Y-%m-%d")
event_window["event_date"] = pd.to_datetime(event_window["event_date"], errors="coerce", utc=True).dt.strftime("%Y-%m-%d")

event_mapping = event_window[["date", "event_date", "event"]].drop_duplicates()

merged_df = daily_oil_price.merge(event_mapping, on="date", how="inner")

target_cols = ["date", "high", "low", "open", "close", "event_date", "event"]
event_oil_price = merged_df[target_cols].copy()

price_cols = ["high", "low", "open", "close"]
event_oil_price[price_cols] = event_oil_price[price_cols]

event_oil_price.to_csv("/Users/alvinwang/Desktop/event_oil_price.csv", index=False)